<a href="https://colab.research.google.com/github/arasuezhile/stkProj/blob/dev/historical_validation_forecaster_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# User Prompt:
# "this one is a multi week model generation script .. it uses until today and checks for next three weeks ..
# i want to validate the originality of it .. so can you make sure to alter it so that i can have the model calculate
# until 3 weeks back .. generates the possible values for next three week which would be until 13th june and
# compares it with the actual value which were at that time"

# Script Purpose: historical_validation_forecaster.py
# This script performs a historical backtest to validate the multi-horizon forecasting model.
# It simulates running the forecast 3 weeks in the past and compares the predictions
# against the known actual outcomes for that period.

# In a Google Colab or similar environment, run the following command in a separate
# cell before executing the rest of the script to install the required libraries.
#!pip install --upgrade yfinance pandas "numpy<2.0" pandas_ta scikit-learn "curl_cffi"

import yfinance as yf
import pandas as pd
import pandas_ta as ta
import numpy as np
from sklearn.ensemble import RandomForestRegressor
import warnings
import json
import os
from datetime import datetime, timedelta
import time
import requests

# Suppress warnings for a cleaner output
warnings.filterwarnings('ignore', category=FutureWarning)

def get_stock_data(ticker_symbol, period="max", end_date=None):
    """
    Fetches historical price data up to a specified end date and makes it timezone-naive.
    """
    try:
        stock = yf.Ticker(ticker_symbol)
        hist_data = stock.history(period=period, end=end_date, auto_adjust=True)
        if hist_data.empty:
            print(f"Error: No historical data found for ticker '{ticker_symbol}'.")
            return None

        # Remove timezone information to prevent comparison errors
        hist_data.index = hist_data.index.tz_localize(None)

        return hist_data
    except Exception as e:
        print(f"An error occurred while fetching data for {ticker_symbol}: {e}")
        return None

def create_weekly_technical_features(daily_df, horizon=1):
    """
    Resamples daily data to weekly and calculates technical indicators
    with parameters tailored to the forecast horizon.
    """
    if daily_df is None:
        return None

    weekly_df = daily_df.resample('W-MON').agg({
        'Open': 'first', 'High': 'max', 'Low': 'min',
        'Close': 'last', 'Volume': 'sum'
    }).dropna()

    if horizon == 1:
        sma_lengths = [5, 10]
        rsi_length = 10
        macd_fast, macd_slow, macd_signal = 8, 16, 6
    elif horizon == 2:
        sma_lengths = [10, 20]
        rsi_length = 14
        macd_fast, macd_slow, macd_signal = 12, 26, 9
    else:
        sma_lengths = [15, 30]
        rsi_length = 20
        macd_fast, macd_slow, macd_signal = 16, 32, 12

    weekly_df.ta.sma(length=sma_lengths[0], append=True)
    weekly_df.ta.sma(length=sma_lengths[1], append=True)
    weekly_df.ta.rsi(length=rsi_length, append=True)
    weekly_df.ta.macd(fast=macd_fast, slow=macd_slow, signal=macd_signal, append=True)
    weekly_df.ta.bbands(length=20, append=True)
    weekly_df.ta.obv(append=True)

    weekly_df.rename(columns={
        f'SMA_{sma_lengths[0]}': f'sma_short', f'SMA_{sma_lengths[1]}': f'sma_long',
        f'RSI_{rsi_length}': 'rsi', f'MACD_{macd_fast}_{macd_slow}_{macd_signal}': 'macd',
        f'MACDh_{macd_fast}_{macd_slow}_{macd_signal}': 'macd_h', f'MACDs_{macd_fast}_{macd_slow}_{macd_signal}': 'macd_s',
        'BBL_20_2.0': 'bb_low', 'BBM_20_2.0': 'bb_mid', 'BBU_20_2.0': 'bb_high',
        'BBB_20_2.0': 'bb_band', 'BBP_20_2.0': 'bb_percent', 'OBV': 'obv'
    }, inplace=True)

    weekly_df.dropna(inplace=True)
    return weekly_df

def train_and_predict(training_data, forecast_data, features):
    """
    A helper function to train a model and make a single prediction.
    Now correctly returns two values.
    """
    if len(training_data) < 1:
        return None, None

    X_train = training_data[features]
    y_train = training_data['target']

    model = RandomForestRegressor(n_estimators=100, random_state=42, min_samples_split=10)
    model.fit(X_train, y_train)

    X_forecast = forecast_data[features]
    predicted_change = model.predict(X_forecast)[0]

    # Create and return feature importances along with the prediction
    feature_importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)

    return predicted_change, feature_importances

def perform_historical_validation(ticker):
    """
    Performs a backtest on a recent historical period to validate the model's originality.
    """
    print(f"--- Performing Historical Validation for {ticker} ---")

    # 1. Define the validation period
    validation_end_date = datetime(2025, 6, 13)
    training_end_date = validation_end_date - timedelta(weeks=3)

    print(f"Simulating 'today' as: {training_end_date.strftime('%Y-%m-%d')}")
    print(f"Forecasting for the next 3 weeks until: {validation_end_date.strftime('%Y-%m-%d')}")

    # 2. Get data for training and the full validation period
    training_daily_data = get_stock_data(ticker, end_date=training_end_date)
    full_daily_data = get_stock_data(ticker, end_date=validation_end_date)

    if training_daily_data is None or full_daily_data is None:
        print("Could not fetch necessary data for validation.")
        return

    all_predictions = {}

    # 3. Generate forecasts as if it were 3 weeks ago
    for week_horizon in range(1, 4):
        weekly_data_train = create_weekly_technical_features(training_daily_data, horizon=week_horizon)
        if weekly_data_train is None or len(weekly_data_train) < 30:
            continue

        # ADDED: Print the last 5 rows of the data used for prediction to allow for verification
        print(f"\n--- Data check for {week_horizon}-Week Model (last 5 weeks of training data) ---")
        print(weekly_data_train.tail())

        temp_data = weekly_data_train.copy()
        temp_data['target'] = (temp_data['Close'].shift(-week_horizon) - temp_data['Close']) / temp_data['Close'] * 100

        train_set = temp_data.iloc[:-week_horizon].dropna(subset=['target'])
        forecast_input = temp_data.iloc[-1:].copy()
        features = [col for col in temp_data.columns if col not in ['Open', 'High', 'Low', 'Close', 'Volume', 'target']]

        # Correctly unpack the two returned values
        pred_change, _ = train_and_predict(train_set, forecast_input, features)

        if pred_change is not None:
            all_predictions[week_horizon] = pred_change

    if not all_predictions:
        print("\nError: Could not generate forecasts for validation.")
        return

    # 4. Compare predictions with actual results
    print("\n" + "="*50)
    print("  Backtest Validation Report")
    print("="*50)

    start_price_df = create_weekly_technical_features(training_daily_data)
    if start_price_df.empty:
        print("Could not create features for start price calculation.")
        return
    start_price = start_price_df.iloc[-1]['Close']

    for week, predicted_change in all_predictions.items():
        predicted_price = start_price * (1 + predicted_change / 100)

        # Get actual price for the corresponding week
        actual_end_date = training_end_date + timedelta(weeks=week)
        weekly_actual = create_weekly_technical_features(full_daily_data.loc[:actual_end_date])
        actual_price = weekly_actual.iloc[-1]['Close']
        actual_change = ((actual_price - start_price) / start_price) * 100

        print(f"\n--- Week {week} Forecast vs. Actual ---")
        print(f"Predicted Change: {predicted_change:+.2f}% (Target: ₹{predicted_price:.2f})")
        print(f"Actual Change:    {actual_change:+.2f}% (Actual Price: ₹{actual_price:.2f})")

        pred_dir = "UP" if predicted_change > 0 else "DOWN"
        actual_dir = "UP" if actual_change > 0 else "DOWN"

        if pred_dir == actual_dir:
            print("Directional Accuracy: CORRECT")
        else:
            print("Directional Accuracy: INCORRECT")


if __name__ == "__main__":
    # --- USER INPUT ---
    # Change the ticker symbol to analyze a different stock
    ticker_symbol = "HINDZINC.NS"

    perform_historical_validation(ticker_symbol)

--- Performing Historical Validation for HINDZINC.NS ---
Simulating 'today' as: 2025-05-23
Forecasting for the next 3 weeks until: 2025-06-13

--- Data check for 1-Week Model (last 5 weeks of training data) ---
             Open        High         Low       Close    Volume   sma_short  \
Date                                                                          
2025-04-28  453.0  467.899994  437.500000  452.500000  14134305  436.929999   
2025-05-05  452.0  455.500000  426.500000  428.250000   5341872  430.200000   
2025-05-12  430.0  436.399994  398.000000  434.950012   9186224  435.840002   
2025-05-19  435.0  455.899994  427.000000  448.200012  10176913  443.270007   
2025-05-26  453.5  455.200012  433.700012  442.399994   7096989  441.260004   

              sma_long        rsi      macd    macd_h    macd_s      bb_low  \
Date                                                                          
2025-04-28  430.285001  52.014235 -4.491247  4.747801 -9.239047  388.510024  